# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [3]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [4]:
load_dotenv("config.env")

True

### VectorDB Instance

In [5]:
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [6]:
from openai import OpenAI

openai_client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),
    base_url=os.environ.get("OPENAI_BASE_URL")
)
print("OpenAI client ready")

OpenAI client ready


In [7]:
# Delete existing collection to start fresh
try:
    chroma_client.delete_collection("udaplay")
    print("Removed existing collection")
except:
    pass

collection = chroma_client.create_collection(name="udaplay")
print(f"Collection '{collection.name}' created")

Removed existing collection
Collection 'udaplay' created


### Add documents

In [8]:
data_dir = "games"
count = 0

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"
    doc_id = os.path.splitext(file_name)[0]

    response = openai_client.embeddings.create(
        input=content,
        model="text-embedding-ada-002"
    )
    embedding = response.data[0].embedding

    collection.add(
        ids=[doc_id],
        documents=[content],
        embeddings=[embedding],
        metadatas=[game]
    )
    count += 1

print(f"✓ Loaded {count} games into ChromaDB")

✓ Loaded 15 games into ChromaDB


In [ ]:
# === Semantic Search Validation ===
print("=== Semantic Search Validation ===\n")

test_query = "When was Pokémon Gold and Silver released?"

query_response = openai_client.embeddings.create(
    input=test_query,
    model="text-embedding-ada-002"
)
query_embedding = query_response.data[0].embedding

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    include=['documents', 'metadatas', 'distances']
)

print(f"Query: '{test_query}'\n")
print("Top 3 Semantic Search Results:")
print("-" * 50)
for i, (doc, meta, dist) in enumerate(zip(
    results['documents'][0],
    results['metadatas'][0],
    results['distances'][0]
), 1):
    similarity = round(1 - dist, 3)
    print(f"\nResult #{i} (Similarity Score: {similarity})")
    print(f"  Game    : {meta.get('Name', 'N/A')}")
    print(f"  Platform: {meta.get('Platform', 'N/A')}")
    print(f"  Year    : {meta.get('YearOfRelease', 'N/A')}")
    print(f"  Preview : {str(doc)[:120]}...")

print("\n✅ Semantic search validation complete!")